# 🎙️ VoiceBatch Studio - Original Self-Built TTS
यह आपका अपना खुद का मॉडल है। यह 100% फ्री है और इसमें कोई 'Chatterbox' या 'Perth' का झंझट नहीं है।

In [ ]:
# @title 📥 Step 1: ज़रूरी इंजन इंस्टॉल करें
print("⏳ वॉइस इंजन तैयार हो रहा है...")
!pip install -q gradio edge-tts librosa soundfile
print("✅ इंजन तैयार है!")

In [ ]:
# @title 🚀 Step 2: VoiceBatch Studio लॉन्च करें
import gradio as gr
import asyncio
import edge_tts
import tempfile
import os

# Realistic आवाज़ बनाने का फंक्शन
async def generate_voice(text, voice_name, rate, pitch):
    # Rate और Pitch को सही फॉर्मेट में बदलना
    r = f"{rate:+}%"
    p = f"{pitch:+}Hz"
    
    communicate = edge_tts.Communicate(text, voice_name, rate=r, pitch=p)
    tmp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
    await communicate.save(tmp_file.name)
    return tmp_file.name

async def get_all_voices():
    v = await edge_tts.VoicesManager.create()
    # केवल अच्छी और साफ़ आवाज़ें (Hindi & English) फिल्टर करना
    return sorted([f"{v['ShortName']}" for v in v.voices])

def launch_app():
    loop = asyncio.get_event_loop()
    voices = loop.run_until_complete(get_all_voices())
    
    with gr.Blocks(theme=gr.themes.Soft()) as demo:
        gr.Markdown("# 🎙️ **VoiceBatch Studio**")
        gr.Markdown("### *अपना खुद का Realistic TTS इंजन (Lifetime Free)*")
        
        with gr.Row():
            with gr.Column():
                input_text = gr.Textbox(label="आपका टेक्स्ट यहाँ लिखें", lines=5, placeholder="नमस्ते, वॉइसबैच स्टूडियो में आपका स्वागत है...")
                voice_dropdown = gr.Dropdown(choices=voices, label="Realistic आवाज़ चुनें", value="hi-IN-MadhurNeural")
                
                with gr.Accordion("आवाज़ की सेटिंग (Realistic Controls)", open=False):
                    speed = gr.Slider(minimum=-50, maximum=50, value=0, label="बोलने की रफ़्तार (Speed)")
                    tone = gr.Slider(minimum=-20, maximum=20, value=0, label="आवाज़ का भारीपन (Pitch)")
            
            with gr.Column():
                output_audio = gr.Audio(label="तैयार आवाज़ (Downloadable)")
                submit_btn = gr.Button("आवाज़ बनाएँ 🚀", variant="primary")
        
        submit_btn.click(generate_voice, inputs=[input_text, voice_dropdown, speed, tone], outputs=output_audio)
        
    demo.launch(share=True)

if __name__ == "__main__":
    launch_app()